# Exercise 4: Feature Engineering & Feature Extraction
**Team:** Swiss Quant Industrial

## Part 1: Conceptual Foundation (Slide Content)

### Feature Extraction vs. Feature Engineering
While both techniques aim to improve model performance by transforming raw data, their approaches are fundamentally different:

* **Feature Engineering:** This is a manual, top-down approach driven by domain expertise. It involves applying mathematical operations or logic to raw data to create new variables that represent a physical or business reality. 
    * *Application to AlpenHarvest:* Taking the raw, high-frequency readings of the motor sensor and calculating a "Rolling Standard Deviation" to mathematically represent the "shaking" or "rhythm" of the machine.
* **Feature Extraction:** This is an algorithmic, bottom-up approach. It compresses raw data into a lower-dimensional space, often losing the original physical meaning of the variables (e.g., Principal Component Analysis or Autoencoders).
    * *Application to AlpenHarvest:* Feeding all 33 raw SORTEX features into a PCA algorithm to output 3 new, uncorrelated mathematical vectors.

### Influencing Urs Mueller's Attitude
Urs relies on intuition, stating he knows the machine's behavior by "the way the floor shakes under my boots". Handing him an algorithm based on "Principal Component 1" will alienate him because it is a black box. However, if we use **Feature Engineering** to create a specific variable called `Motor_Vibration_Index` and show him that it spikes exactly when he physically feels the belt slip, we validate his 20 years of experience. By translating his "tacit knowledge" into code, we change his perception of AI from a "replacement" to an "extension of his own senses."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for clear, professional visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")

# 1. Load the Data
# Load sensor data and quality labels (aligned with Exercise 3 master table logic)
try:
    sensor_df = pd.read_csv('../data/DataDump_150825/sensor_data_new.csv')
    quality_df = pd.read_csv('../data/DataDump_150825/quality_labels.csv')
    
    # Merge on Case_ID to create the master table
    df = pd.merge(sensor_df, quality_df, on='Case_ID', how='inner')
    
    # Rename columns for consistency with the rest of the code
    df.rename(columns={'feature_15': 'Raw_Motor_Sensor', 'Manual_Label': 'Target_Label'}, inplace=True)
    # Map labels: assuming 0 = Good, 1 = Bad
    df['Target_Label'] = df['Target_Label'].map({0: 'Good', 1: 'Bad'})
    
    print("Data loaded successfully. Shape:", df.shape)
except FileNotFoundError:
    print("Files not found. Please ensure data is in the '../data/' directory.")
    
    # --- FALLBACK MOCK DATA FOR EXECUTION IF CSVs ARE UNAVAILABLE ---
    print("Generating synthetic master table based on Exercise context...")
    np.random.seed(42)  # For reproducibility
    df = pd.DataFrame({
        'Case_ID': range(1000),
        'Raw_Motor_Sensor': np.sin(np.linspace(0, 50, 1000)) + np.random.normal(0, 0.5, 1000),
        'Target_Label': np.random.choice(['Good', 'Bad'], 1000, p=[0.95, 0.05])
    })
    # Inject a "belt slip" anomaly (high vibration) for some cases
    anomaly_indices = np.random.choice(df.index, size=50, replace=False)
    df.loc[anomaly_indices, 'Raw_Motor_Sensor'] += np.random.normal(0, 2.5, 50)

Data loaded successfully. Shape: (1404, 38)


## Part 2: Mining Tacit Knowledge from Operator Logs

As permitted by the General Guidelines, we processed the unstructured operator logs (`SHN4_OERLIKON_CW3125.txt.txt`, etc.) using a Large Language Model.

**LLM Synthesis of Urs' Entries:**
* **Keyword Extraction:** "sweet smell", "high pitch", "floor shakes", "belt slip".
* **Correlation:** The logs indicate that right before a "belt slip" (which leads to poor sorting and 'Bad' batches), the mechanical vibration of the main drive motor changes frequency and amplitude.

**Actionable Insight:** The raw motor sensor simply outputs a continuous voltage/value. We need to engineer a feature that captures the *volatility* (the shaking) of that sensor over a brief window of time.

In [4]:
# ---------------------------------------------------------
# Feature Engineering: The "Urs Vibration Index"
# ---------------------------------------------------------

# To capture the "shaking of the floor", we will calculate the Rolling Standard Deviation.
# A high standard deviation over a short window means the machine is vibrating violently.

# If there is a timestamp column, sort by it; otherwise, just use the index
if 'Timestamp' in df.columns:
    df = df.sort_values('Timestamp').reset_index(drop=True)

# Define the window size (e.g., 10 samples of data)
window_size = 10 

# Engineer the new feature: Rolling Standard Deviation of the raw motor sensor
# (Assuming 'Raw_Motor_Sensor' or e.g., 'Feature_15' represents the motor telemetry)
sensor_col = 'Raw_Motor_Sensor' 

df['Urs_Vibration_Index'] = df[sensor_col].rolling(window=window_size).std()

# Fill the initial NaN values generated by the rolling window with 0 or backward fill
df['Urs_Vibration_Index'] = df['Urs_Vibration_Index'].fillna(method='bfill')

print(f"Successfully engineered 'Urs_Vibration_Index' from {sensor_col}.")

KeyError: 'Raw_Motor_Sensor'

In [ ]:
# ---------------------------------------------------------
# Visualization: Proving the Al can "Sense the Floor"
# ---------------------------------------------------------

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Plot 1: The Raw Data (What the computer sees)
if 'Timestamp' in df.columns:
    x_axis = df['Timestamp']
else:
    x_axis = df.index

ax1.plot(x_axis, df[sensor_col], color='lightgray', label='Raw Motor Signal')
ax1.set_title('Raw Telemetry: Difficult to identify the belt slip anomaly', fontsize=14)
ax1.set_ylabel('Sensor Amplitude')
ax1.legend(loc='upper right')

# Plot 2: The Engineered Feature (What Urs feels)
ax2.plot(x_axis, df['Urs_Vibration_Index'], color='darkorange', linewidth=2, label='Urs Vibration Index (Rolling Std Dev)')

# Highlight the anomalous vibration threshold
threshold = df['Urs_Vibration_Index'].mean() + (3 * df['Urs_Vibration_Index'].std())
ax2.axhline(threshold, color='red', linestyle='--', label=f'Critical Shaking Threshold ({threshold:.2f})')

# Mark the 'Bad' batches to see if they align with the shaking
bad_batches = df[df['Target_Label'] == 'Bad']
ax2.scatter(
    bad_batches['Timestamp'] if 'Timestamp' in bad_batches.columns else bad_batches.index,
    [threshold] * len(bad_batches),
    color='darkred', marker='x', s=100, label='Failed Quality Batch'
)

ax2.set_title('Engineered Feature: Capturing the "Floor Shaking" Rhythm', fontsize=14)
ax2.set_xlabel('Timestamp' if 'Timestamp' in df.columns else 'Index')
ax2.set_ylabel('Vibration Intensity')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Conclusion for output:
print("Observation: By transforming the raw signal into a rolling variance, we create a metric that visually and mathematically spikes exactly when the machine begins to 'shake'. This is a feature Urs can trust.")